# Gold Sales Layer
Builds the analytics-ready gold layer in `sales_data.gold` from `sales_data.silver`: a star schema (**4 dimensions**, **3 facts**) plus **3 summary marts** for reporting.

In [0]:
CATALOG = "sales_data"
GOLD = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
print(f"Schema ready: {GOLD}")

In [0]:
# Dimension tables from silver (SCD Type 1 - current state)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_customer AS
SELECT customer_id, customer_code, first_name, last_name, email, city, state, country,
       loyalty_tier, signup_date, is_active
FROM {CATALOG}.silver.customers
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_product AS
SELECT product_id, sku, product_name, category, brand, unit_price, unit_cost,
       ROUND((unit_price - unit_cost) / unit_price * 100, 2) AS margin_pct, is_active
FROM {CATALOG}.silver.products
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_store AS
SELECT store_id, store_code, store_name, region, city, state, store_type, open_date
FROM {CATALOG}.silver.stores
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_promotion AS
SELECT promo_id, promo_code, promo_type, discount_value, channel, start_date, end_date,
       DATEDIFF(end_date, start_date) AS duration_days
FROM {CATALOG}.silver.promotions
""")

print("Dimensions created: dim_customer, dim_product, dim_store, dim_promotion")

In [0]:
# Fact tables from silver (line_total is already the net amount:
# quantity * unit_price_sold * (1 - discount_pct), verified against silver)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_orders AS
SELECT o.order_id, o.order_date,
       YEAR(o.order_date) AS order_year, MONTH(o.order_date) AS order_month,
       o.customer_id, o.store_id, o.promo_id, o.channel, o.order_status, o.payment_method,
       o.order_total, o.item_count
FROM {CATALOG}.silver.orders o
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_order_items AS
SELECT i.item_id, i.order_id, i.order_date, i.product_id, i.quantity,
       i.unit_price_sold, i.discount_pct, i.line_total
FROM {CATALOG}.silver.order_items i
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_returns AS
SELECT r.return_id, r.order_id, r.item_id, r.order_date, r.return_date,
       r.returned_quantity, r.refund_amount, r.return_reason, r.return_status
FROM {CATALOG}.silver.returns r
""")

print("Facts created: fact_orders, fact_order_items, fact_returns")

In [0]:
# Mart 1: daily sales by region
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.agg_daily_sales_summary AS
SELECT o.order_date, s.region,
       COUNT(DISTINCT o.order_id) AS total_orders,
       ROUND(SUM(o.order_total), 2) AS total_revenue,
       ROUND(AVG(o.order_total), 2) AS avg_order_value,
       SUM(o.item_count) AS total_units
FROM {GOLD}.fact_orders o
JOIN {GOLD}.dim_store s ON o.store_id = s.store_id
GROUP BY o.order_date, s.region
""")

# Mart 2: monthly sales by product category
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.agg_monthly_category_sales AS
SELECT DATE_TRUNC('month', i.order_date) AS sales_month, p.category,
       COUNT(DISTINCT i.order_id) AS orders_containing,
       SUM(i.quantity) AS units_sold,
       ROUND(SUM(i.line_total), 2) AS revenue
FROM {GOLD}.fact_order_items i
JOIN {GOLD}.dim_product p ON i.product_id = p.product_id
GROUP BY DATE_TRUNC('month', i.order_date), p.category
ORDER BY sales_month
""")

# Mart 3: customer-level sales summary
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.agg_customer_sales_summary AS
SELECT c.customer_id, c.first_name, c.last_name, c.loyalty_tier,
       COUNT(o.order_id) AS total_orders,
       ROUND(SUM(o.order_total), 2) AS total_spend,
       ROUND(AVG(o.order_total), 2) AS avg_order_value,
       MIN(o.order_date) AS first_order_date,
       MAX(o.order_date) AS last_order_date
FROM {GOLD}.fact_orders o
JOIN {GOLD}.dim_customer c ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.loyalty_tier
""")

print("Marts created: agg_daily_sales_summary, agg_monthly_category_sales, agg_customer_sales_summary")

In [0]:
from pyspark.sql import functions as F

GOLD_TABLES = ["dim_customer", "dim_product", "dim_store", "dim_promotion",
               "fact_orders", "fact_order_items", "fact_returns",
               "agg_daily_sales_summary", "agg_monthly_category_sales", "agg_customer_sales_summary"]

counts = {t: spark.table(f"{GOLD}.{t}").count() for t in GOLD_TABLES}
print("Row counts:")
for t, n in counts.items():
    print(f"  {t}: {n:,}")

def orphan(fact, fk, dim, pk):
    return spark.sql(f"""
        SELECT COUNT(*) FROM {GOLD}.{fact} f
        LEFT ANTI JOIN {GOLD}.{dim} d ON f.{fk} = d.{pk}
    """).first()[0]

rows = []
def add_check(name, expected, actual):
    rows.append({"check": name, "expected": float(expected), "actual": float(actual),
                 "status": "PASS" if expected == actual else "FAIL"})

# Row counts reconcile with silver
add_check("fact_orders rows vs silver.orders", counts["fact_orders"],
          spark.table(f"{CATALOG}.silver.orders").count())
add_check("fact_order_items rows vs silver.order_items", counts["fact_order_items"],
          spark.table(f"{CATALOG}.silver.order_items").count())
add_check("fact_returns rows vs silver.returns", counts["fact_returns"],
          spark.table(f"{CATALOG}.silver.returns").count())

# Referential integrity: orphan foreign keys must be 0
add_check("orphan fact_orders.customer_id", 0, orphan("fact_orders", "customer_id", "dim_customer", "customer_id"))
add_check("orphan fact_orders.store_id", 0, orphan("fact_orders", "store_id", "dim_store", "store_id"))
add_check("orphan fact_order_items.order_id", 0, orphan("fact_order_items", "order_id", "fact_orders", "order_id"))
add_check("orphan fact_order_items.product_id", 0, orphan("fact_order_items", "product_id", "dim_product", "product_id"))
add_check("orphan fact_returns.item_id", 0, orphan("fact_returns", "item_id", "fact_order_items", "item_id"))
add_check("orphan fact_returns.order_id", 0, orphan("fact_returns", "order_id", "fact_orders", "order_id"))

# Mart reconciliation: mart totals must match the facts
silver_rev = spark.table(f"{CATALOG}.silver.orders").agg(F.round(F.sum("order_total"), 2)).first()[0]
daily_rev = spark.table(f"{GOLD}.agg_daily_sales_summary").agg(F.round(F.sum("total_revenue"), 2)).first()[0]
cust_rev = spark.table(f"{GOLD}.agg_customer_sales_summary").agg(F.round(F.sum("total_spend"), 2)).first()[0]
add_check("agg_daily_sales_summary revenue vs silver orders", silver_rev, daily_rev)
add_check("agg_customer_sales_summary revenue vs silver orders", silver_rev, cust_rev)

items_rev = spark.table(f"{GOLD}.fact_order_items").agg(F.round(F.sum("line_total"), 2)).first()[0]
cat_rev = spark.table(f"{GOLD}.agg_monthly_category_sales").agg(F.round(F.sum("revenue"), 2)).first()[0]
add_check("agg_monthly_category_sales revenue vs fact_order_items", items_rev, cat_rev)

display(spark.createDataFrame(rows))